# NB4 - Engines + deployment pipeline (image in -> per-worker risk out)   [v3: tuned detector settings, gated PPE prior, evidence-aware aggregation]
Takes an image together with the saved outputs of the other notebooks and returns, for every worker: PPE, struck-by and fall scores, one combined risk score, a risk band, the distance to the nearest machine (with **either** distance method, switchable) and how high above the ground the worker is.

This is the deployment notebook only. Evaluation, ablations and the distance-accuracy study live in their own notebooks. A Streamlit / Gradio wrapper comes later and will call the same functions.

## What this notebook expects (Google Drive layout)

```
MyDrive/cv5570_risk/
   weights/   det_ppe_worker_best.pt, class_thresholds_ppe_worker.json, calib_ppe_worker.joblib   <- fine-tuning notebook
              risk_config.json                                                                      <- NB7 (diagnostics): machine threshold, input size, PPE priors by head size. Optional: safe defaults are used (with a warning) if it is missing.
   code/      risk_pose.py                                                                          <- written by the POSE notebook (run it once)
   <TARGET>/              'demo' by default; 'accuracy' also works
      images/             the images to analyse
      depth/<name>.npz    Depth Pro notebook
      camera/<name>.json  GeoCalib notebook
      pose/<name>.json    pose notebook (OPTIONAL cache: if missing, pose is run here)
      camera_heights.json OPTIONAL  {"name": 1.5}  camera height above the floor in metres, if known
      zones.json          OPTIONAL  manual fall zones: {"name": [{"polygon": [[x,y],...], "drop_m": 3.0}]}
      results/            written here: results.csv / results.json and annotated images
```
* All outputs are matched to an image by its file name (without extension), so the same photos must have been run through the other notebooks.
* If one distance method's output is missing for an image, that method falls back to a rough pixel-based estimate and the notebook says so.
* **If you change the pose settings:** edit them in the pose notebook's module cell and re-run that cell; this notebook reloads the module each time it starts.
* **If you retrain the detector:** re-run NB7 so that `risk_config.json` matches the new weights.
* **Optional second machine detector (YOLO-World):** used only if `risk_config.json` says it helps (or you force `USE_WORLD` in the config cell). It downloads its weights on first use.

**Detector classes** (read from the fine-tuned model) and how they are used:

| model class | role in the pipeline |
|---|---|
| Person | a worker (merged with the pose model's people) |
| Hardhat / NO-Hardhat | helmet present / helmet missing evidence (calibrated probability used directly, no hard threshold) |
| Safety Vest / NO-Safety Vest | vest present / vest missing evidence (same) |
| machinery, vehicle | **one merged "machine" class** (struck-by hazard), plus optional YOLO-World boxes for machine types MOCS lacks |
| Mask, NO-Mask, Safety Cone | detected but **not scored** |

## How the scores work: the engines and where the weights come from

Every worker gets three engine scores (0-100). They are combined into one risk score. Nothing below is learned from your images: the formulas are fixed and the weights come from fatality statistics.

### Who counts as a worker
One list: every person found by the **full-frame pose model** (with 17 body joints), plus any person the detector found that the pose model missed (no joints, so posture and elevation are `unknown` for those). Pose people and detector people are matched by overlap, one-to-one.

### PPE engine - is this worker missing a helmet or vest?
* **Inputs:** the helmet / vest / NO-helmet / NO-vest boxes assigned to the worker. Each PPE box goes to exactly one worker: the smallest worker box that contains its centre, in the head zone (top 40%) for helmets and the torso zone (10-75% of the height) for vests. Detector confidences are first **calibrated** (isotonic regression fitted on the validation set) and then used **directly**: there is no hard cut-off, so a weak helmet detection counts as weak evidence instead of being thrown away.
* **Per item:** `m = (1 - p_item) x (p_NO + (1 - p_NO) x prior)`. Item clearly seen: m is about 0. A "NO-item" box seen and no item: m is about 1. **Nothing seen either way: m = prior**, where the prior is the measured rate at which "no box found" really means "missing" (NB7, on MOCS validation), looked up by how big the worker's head is in pixels. Big head and no helmet found: the prior is high. Tiny head: see next bullet.
* **Unverifiable:** if the head is smaller than the size at which helmets are found at all (NB7 finds it, about 16 px at detector resolution) *and* nothing about this worker's PPE was detected, the PPE score is **not computed** (`n/a`, status `unverifiable`) instead of guessing. The worker is still listed, with low risk confidence.
* **Score** = `100 x [0.62 x (1 + 0.5 x proximity) x m_helmet + 0.19 x (1 + proximity) x m_vest]`, capped at 100, where proximity is the struck-by score / 100. Missing PPE next to a machine counts up to 1.5x (helmet) and 2x (vest).
* **Output:** score, `ppe_status` (`evidence` = something was detected, `prior` = nothing detected, prior used, `unverifiable`), and which item(s) look missing.
* **Honesty:** `0` is never reached (that needs both items detected with probability 1), `~3-6` is a well-detected compliant worker, `~65-100` is likely-missing PPE (higher near a machine).

### Struck-by engine - is this worker close to a machine?
* **Inputs:** the worker's feet, each machine's bottom edge, and the chosen distance method (geometry or Depth Pro).
* **Distance:** metres between the worker's feet and the machine's base. A worker standing on a platform has the feet placed on *their own* plane (not projected down to the ground), so the distance is not exaggerated.
* **Per machine:** `s = exp(-(d / R)^2)` with `R = max(3 m, machine width in metres)`. A bigger machine has a bigger danger zone (swing, reach), never smaller than 3 m.
* **Several machines:** combined as `1 - (1 - s1)(1 - s2)...`, so more machines can only raise the score. Score = 100 x that.
* **Honesty flags:** if no machine is detected the score is 0 but is flagged `no_machines_found` (not "safe": machine recall is limited, and the detector may not know this kind of machine). Distance confidence is `high` (3+ ground workers gave a consistent camera height, or the camera height is known), `medium`, or `low` (rough pixel fallback, or a worker whose feet cannot be placed).

### Fall engine - is this worker high up, and unstable?
* **Height above the ground (elevation):** every upright, full-body worker acts as a 1.7 m ruler that tells how far the camera is above the plane they stand on. Workers on the ground agree with each other (that group is the ground); a worker on a scaffold or platform shows a smaller camera height, and the difference is their elevation. Feet at or above the camera's own eye level mean "elevated by about the camera height or more". Verdicts need at least two ground workers (or a known camera height) within about 20 m; otherwise elevation is `unknown` and is not scored. **Severity = min(1, elevation / 1.8 m)**, because fall protection is required from 6 ft (29 CFR 1926.501).
* **Posture** (from the joints): torso lean (0 deg upright to 90 deg horizontal; counts from 20 deg, full at 70 deg), balance (hip centre horizontally outside the ankles, relative to body height), and a "down" state when the person box is wider than tall. No joints: posture is `unknown` and adds nothing.
* **Score** = `100 x (1 - (1 - 0.6 x severity)(1 - 0.8 x posture))`: being high, or being unstable, raises it; both together raise it most. Optional hand-drawn zones (`zones.json`) can add exposure; the larger of the two severities is used.
* **What it does not know:** whether a guardrail or harness is present. It flags *exposure to a fall from height*, not an unprotected edge. Posture thresholds are starting values and have not been tuned on a fall dataset.

### Combining the three
`R = 0.46 x Fall + 0.29 x PPE + 0.25 x Struck-by + 0.15 x (F.P + F.S + P.S)`, with scores on 0-1, then `R` is raised to at least 0.7 x the worst single engine **that has real evidence behind it** (PPE: something was detected; struck-by: a machine was found; fall: elevation was judged or a zone applies). A score that is only a prior or a default can no longer set the floor, which is what made most workers sit at the same value before. Every row also gets a **risk confidence** (high / medium / low = how many engines have real evidence).

**Bands:** Low / Medium / High / Critical at 25 / 50 / 75 are **placeholders**. Once you have a few dozen workers, the optional last cell turns them into quantiles of your own results (50th / 75th / 90th percentile) and saves them to `risk_config.json`. Those bands are *relative* (they rank workers against this set of photos); the raw `risk` and a `risk_percentile` column are always saved too.

### Where 0.46 / 0.29 / 0.25 come from
US construction fatalities, BLS 2023 (1,075 deaths), each cause mapped to the engine that addresses it:

| Engine | Evidence | Share of deaths |
|---|---|---|
| Fall | falls, slips and trips: 421 deaths | 39.2% |
| Struck-by | contact incidents 13.8%, plus the on-foot share of transportation incidents (22.3% x 68%, where 68% = 52.7 / (52.7 + 24.8) from road-work-zone data) = 13.8 to 29.0%; midpoint used | 21.4% |
| PPE | traumatic brain injuries (NIOSH, 2003-10) | 25.0% |

Normalising 39.2 : 21.4 : 25.0 gives **fall 0.46, struck-by 0.25, PPE 0.29**. Across the struck-by range they move between 0.42-0.50 (fall), 0.18-0.31 (struck-by), 0.27-0.32 (PPE); report that range as a sensitivity result.

**Inside the PPE engine:** a helmet addresses the 25% head-injury share; a vest addresses visibility to vehicle operators, 22.3% x 68% = 15.2% near machinery and half of that elsewhere. Normalised over 25 + 15.2: **helmet 0.62**, **vest 0.19**, rising to 0.38 as proximity goes to 1, which is the `(1 + proximity)` factor.

**Caveats to state in the report:**
* The road-work shares are borrowed for general sites; head injuries overlap with falls; fatality shares are not per-exposure rates. The weights are *priors for ranking*, not probabilities.
* `0.15` (pairwise term), the `0.7` floor, the 3 m minimum radius, the `0.6` / `0.8` weights inside the fall score, the `0.5` helmet-proximity factor, the posture thresholds and the band cut-offs are **not** derivable from any table. The PPE priors and the machine threshold *are* measured (NB7), on MOCS. They are sensitivity parameters; band cut-offs should eventually come from quantiles on a test set.
* Mask is not scored (a chronic hazard, absent from the fatality tables). Machine type is not weighted (no data to back a multiplier).

Sources: [BLS CFOI 2023](https://www.bls.gov/news.release/archives/cfoi_12192024.htm) | [NIOSH, construction helmets and TBI](https://cdc.gov/niosh/bulletin/2022/construction-helmets.html) | [Work Zone Safety Clearinghouse](https://workzonesafety.org/work-zone-data/worker-fatalities-and-injuries-at-road-construction-sites). Re-check the figures against these before they go into the report.

In [ ]:
!pip -q install "ultralytics>=8.4.0"
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, glob, json, math, importlib
import numpy as np, pandas as pd, cv2, joblib
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT   = '/content/drive/MyDrive/cv5570_risk'
TARGET = 'demo'                      # 'demo' or 'accuracy' (the folder that holds images/ depth/ camera/ [pose/])
W_DIR  = f'{ROOT}/weights'
DET_WEIGHTS = f'{W_DIR}/det_ppe_worker_best.pt'
THR_JSON    = f'{W_DIR}/class_thresholds_ppe_worker.json'
CALIB_PKL   = f'{W_DIR}/calib_ppe_worker.joblib'

BASE = f'{ROOT}/{TARGET}'
IMG_DIR, DEPTH_DIR, CAM_DIR, POSE_DIR = f'{BASE}/images', f'{BASE}/depth', f'{BASE}/camera', f'{BASE}/pose'
OUT_DIR = f'{BASE}/results'
for d in (IMG_DIR, OUT_DIR): os.makedirs(d, exist_ok=True)

# shared pose module (written to Drive by the pose notebook)
assert os.path.exists(f'{ROOT}/code/risk_pose.py'), 'Run the pose notebook once first: it writes code/risk_pose.py to Drive.'
sys.path.insert(0, f'{ROOT}/code')
import risk_pose as rp; importlib.reload(rp)

# measured settings from NB7 (safe defaults if the file is missing)
USE_WORLD = None                 # None = follow risk_config.json; True / False forces the YOLO-World second machine detector on / off
USE_QUANTILE_BANDS = True        # use the band edges saved in risk_config.json (if any) instead of 25/50/75
DEFAULTS = dict(det_imgsz=1280, machine_thr=0.2, world=dict(use=False, thr=0.2, model='yolov8s-worldv2.pt',
                prompts=['concrete pump truck', 'crane', 'excavator', 'dump truck', 'loader', 'bulldozer', 'truck', 'construction machine']),
                person_thr=None, head_bins=[0, 10, 16, 24, 36, 1e9], prior_helmet=[0.5] * 5, prior_vest=[0.5] * 5, unverifiable_below_px=0.0, band_edges=None)
CONFIG_PATH = f'{W_DIR}/risk_config.json'
RC = dict(DEFAULTS)
if os.path.exists(CONFIG_PATH): RC.update(json.load(open(CONFIG_PATH))); print('loaded', CONFIG_PATH, '(tuned', RC.get('created'), ')')
else: print('WARNING: risk_config.json not found - using untuned defaults. Run NB7 (diagnostics) first for measured settings.')
if USE_WORLD is not None: RC['world']['use'] = USE_WORLD

CFG = dict(
    # aggregator weights (BLS fatality shares, see the text above)
    W_FALL=0.46, W_PPE=0.29, W_STRUCK=0.25, GAMMA=0.15, FLOOR=0.7,
    # PPE engine
    W_HELMET=0.62, W_VEST=0.19, HELMET_PROX=0.5,          # priors by head size come from risk_config.json
    # struck-by engine
    WORKER_H=1.7, MIN_RADIUS=3.0, MIN_MACHINE_AREA=0.005,
    # fall engine (weights are sensitivity parameters; the posture thresholds live in risk_pose.py)
    W_HEIGHT=0.6, W_POSTURE=0.8,
    # bands: placeholders
    BANDS=[(25, 'Low'), (50, 'Medium'), (75, 'High'), (101, 'Critical')],
    AGREE_TOL=25.0,                     # the two distance methods "agree" if within this % of their mean
    DET_IMGSZ=int(RC['det_imgsz']), MACHINE_THR=float(RC['machine_thr']),
)
if USE_QUANTILE_BANDS and RC.get('band_edges'):
    e = RC['band_edges']; CFG['BANDS'] = [(e[0], 'Low'), (e[1], 'Medium'), (e[2], 'High'), (101, 'Critical')]; print('bands from quantiles:', e)

# detector class -> role (exact names of the fine-tuned model)
ROLE = {'Person': 'person', 'Hardhat': 'helmet', 'NO-Hardhat': 'no_helmet', 'Safety Vest': 'vest',
        'NO-Safety Vest': 'no_vest', 'machinery': 'machine', 'vehicle': 'machine'}      # Mask, NO-Mask, Safety Cone: ignored

In [ ]:
# LOAD MODELS
det   = YOLO(DET_WEIGHTS)
THR   = json.load(open(THR_JSON))
CALIB = joblib.load(CALIB_PKL)
ZONES   = json.load(open(f'{BASE}/zones.json')) if os.path.exists(f'{BASE}/zones.json') else {}
H_KNOWN = json.load(open(f'{BASE}/camera_heights.json')) if os.path.exists(f'{BASE}/camera_heights.json') else {}

_pose = None
def pose_model():
    """the pose model is only loaded if some image has no cached pose/<name>.json"""
    global _pose
    if _pose is None: _pose = rp.load_pose_model()
    return _pose

names = list(det.names.values())
print('detector classes:', names)
missing = [c for c in ROLE if c not in names]
assert not missing, f'these classes are not in the model: {missing} - check ROLE'
print('pose settings:', {k: rp.CFG[k] for k in ('POSE_MODEL', 'POSE_IMGSZ', 'MAX_RANGE_M', 'ELEV_MIN_M')}, '| known camera heights:', H_KNOWN or 'none')

## 1. Detection, one worker list, PPE association

In [ ]:
def calibrated(name, c):
    return float(CALIB[name].predict([c])[0]) if name in CALIB else c

PPE_ROLES = ('helmet', 'no_helmet', 'vest', 'no_vest')
_world = None
def world_model():
    global _world
    if _world is None:
        from ultralytics import YOLOWorld
        _world = YOLOWorld(RC['world']['model']); _world.set_classes(RC['world']['prompts'])
    return _world

def nms_boxes(items, thr=0.5):
    keep = []
    for d in sorted(items, key=lambda d: -d['raw']):
        if all(rp.iou(d['box'], k['box']) < thr for k in keep): keep.append(d)
    return keep

def detect(img):
    """fine-tuned detector (+ optional YOLO-World for machines). PPE: calibrated p used directly (no hard threshold).
    Person: class threshold. Machines: machinery + vehicle merged, one measured threshold."""
    r = det.predict(img, imgsz=CFG['DET_IMGSZ'], conf=0.05, verbose=False)[0]
    out, mach = [], []
    for b in r.boxes:
        name = det.names[int(b.cls)]; role = ROLE.get(name); c = float(b.conf)
        if role is None: continue
        d = dict(role=role, name=name, raw=c, p=calibrated(name, c), box=[float(v) for v in b.xyxy[0]])
        if role == 'machine':
            if c >= CFG['MACHINE_THR']: mach.append(d)
        elif role in PPE_ROLES: out.append(d)
        elif c >= ((RC.get('person_thr') or THR.get(name, 0.25)) if role == 'person' else THR.get(name, 0.25)): out.append(d)
    mach = nms_boxes(mach)
    if RC['world']['use']:
        try:
            wr = world_model().predict(img, imgsz=960, conf=RC['world']['thr'], verbose=False)[0]
            for b in wr.boxes:
                w = dict(role='machine', name='world:' + RC['world']['prompts'][int(b.cls)], raw=float(b.conf), p=float(b.conf), box=[float(v) for v in b.xyxy[0]])
                if all(rp.iou(w['box'], m['box']) < 0.5 for m in mach): mach.append(w)
        except Exception as e:
            print('YOLO-World unavailable, using the fine-tuned detector only:', e); RC['world']['use'] = False
    return out + mach

def get_workers(img, stem, dets):
    """one worker list: full-frame pose people (joints cached by the pose notebook if available, else computed here)
    plus detector people the pose model missed"""
    H, W = img.shape[:2]; persons, source = None, 'inline'
    jp = f'{POSE_DIR}/{stem}.json'
    if os.path.exists(jp):
        d = json.load(open(jp))
        if (d['width'], d['height']) == (W, H) and d.get('pose_model') == rp.CFG['POSE_MODEL'] and d.get('imgsz') == rp.CFG['POSE_IMGSZ']:
            persons, source = rp.raw_persons_from_json(d), 'cached'
    if persons is None:
        persons = rp.run_pose(pose_model(), img)
    det_p = [d for d in dets if d['role'] == 'person']
    return rp.merge_detector_persons(persons, [d['box'] for d in det_p], [d['raw'] for d in det_p]), source

def associate(persons, ppe_dets):
    """each PPE box goes to exactly one worker: smallest worker box containing its centre, in a plausible body zone"""
    for p in persons:
        p['ppe'] = dict(helmet=0.0, no_helmet=0.0, vest=0.0, no_vest=0.0)
    for d in ppe_dets:
        cx, cy = (d['box'][0] + d['box'][2]) / 2, (d['box'][1] + d['box'][3]) / 2
        zone = (0.0, 0.40) if d['role'] in ('helmet', 'no_helmet') else (0.10, 0.75)
        cands = []
        for i, p in enumerate(persons):
            x1, y1, x2, y2 = p['box']; h = max(1.0, y2 - y1)
            if x1 <= cx <= x2 and y1 <= cy <= y2 and zone[0] <= (cy - y1) / h <= zone[1]:
                cands.append(((x2 - x1) * h, i))
        if cands:
            i = min(cands)[1]
            persons[i]['ppe'][d['role']] = max(persons[i]['ppe'][d['role']], d['p'])

def filter_machines(dets, shape):
    H, W = shape[:2]
    return [d for d in dets if d['role'] == 'machine'
            and (d['box'][2] - d['box'][0]) * (d['box'][3] - d['box'][1]) >= CFG['MIN_MACHINE_AREA'] * H * W]

feet_px = rp.feet_px

## 2. Distance module: two methods
* **geometry:** GeoCalib gives the camera's focal length and which way is down; the ground workers (1.7 m rulers) give the camera height; feet and machine bases are projected onto their floor planes.
* **depthpro:** Depth Pro's metric depth map, back-projected to 3D.

In [ ]:
_cam_cache, _depth_cache = {}, {}
def load_camera(stem):
    if stem not in _cam_cache:
        p = f'{CAM_DIR}/{stem}.json'
        _cam_cache[stem] = json.load(open(p)) if os.path.exists(p) else None
    return _cam_cache[stem]

def load_depth(stem):
    if stem not in _depth_cache:
        p = f'{DEPTH_DIR}/{stem}.npz'
        if os.path.exists(p):
            z = np.load(p); _depth_cache[stem] = (z['depth'].astype(np.float32), float(z['f_px']))
        else:
            _depth_cache[stem] = None
    return _depth_cache[stem]

def build_ctx(an, method):
    """decides which distance mode is actually usable for this image; 'pixel' is the rough fallback"""
    img, stem, cam, scene = an['img'], an['stem'], an['cam'], an['scene']
    H, W = img.shape[:2]
    ctx = dict(stem=stem, H=H, W=W, method=method, mode='pixel', tier='low', h=None, spread=None, n_ref=0,
               cam=None, depth=None, f_px=None, f_geocalib=None, f_depthpro=None, note=an['cam_note'])
    raw_cam, dep = load_camera(stem), load_depth(stem)
    if raw_cam is not None: ctx['f_geocalib'] = raw_cam['fx']
    if dep is not None:
        ctx['f_depthpro'] = dep[1]
        if dep[0].shape != (H, W): ctx['note'] = 'Depth Pro map size differs from image'; dep = None
    if method == 'geometry':
        if cam is None: ctx['note'] = ctx['note'] or 'no GeoCalib output for this image'
        elif rp.is_topdown(cam): ctx['note'] = 'camera looks almost straight down'
        elif scene['h_ref'] is None: ctx['note'] = scene['status'] or 'no upright full-body worker to set the scale'
        else:
            sp, n = scene['spread'], scene['n_ref']
            known = scene['ref_source'] == 'known camera height'
            ctx.update(mode='geometry', cam=cam, h=scene['h_ref'], spread=sp, n_ref=n,
                       tier='high' if (known or (n >= 3 and sp is not None and sp <= 0.15)) else 'medium')
    elif method == 'depthpro':
        if dep is None: ctx['note'] = ctx['note'] or 'no Depth Pro output for this image'
        else: ctx.update(mode='depthpro', depth=dep[0], f_px=dep[1], tier='medium')
    return ctx

def depth_xyz(u, v, ctx):
    D, f = ctx['depth'], ctx['f_px']; Hd, Wd = D.shape
    ui, vi = int(np.clip(round(u), 0, Wd - 1)), int(np.clip(round(v), 0, Hd - 1))
    Z = float(np.median(D[max(0, vi - 2):vi + 3, max(0, ui - 2):ui + 3]))
    return np.array([(ui - Wd / 2) * Z / f, (vi - Hd / 2) * Z / f, Z])

def machine_pt(u, v, ctx):
    """a machine's base point (machines are assumed to stand on the ground plane)"""
    return rp.floor_point(u, v, ctx['cam'], ctx['h']) if ctx['mode'] == 'geometry' else depth_xyz(u, v, ctx)

def feet3d(wk, ctx):
    fu, fv = feet_px(wk)
    if ctx['mode'] == 'geometry':
        e = wk['elev']
        if e['kind'] == 'above_camera': return None          # feet at/above eye level: no plane to place them on
        h = e['h_i'] if (e['kind'] == 'elevated' and e['h_i'] is not None) else ctx['h']      # elevated: their OWN plane
        return rp.floor_point(fu, fv, ctx['cam'], h)
    return depth_xyz(fu, fv, ctx)

def worker_machine_dist(wk, m, ctx):
    """-> (distance_m, machine_width_m, mode used)"""
    x1, y1, x2, y2 = m['box']
    if ctx['mode'] in ('geometry', 'depthpro'):
        pts = [feet3d(wk, ctx), machine_pt((x1 + x2) / 2, y2, ctx), machine_pt(x1, y2, ctx), machine_pt(x2, y2, ctx)]
        if all(p is not None for p in pts):
            return float(np.linalg.norm(pts[0] - pts[1])), float(np.linalg.norm(pts[2] - pts[3])), ctx['mode']
    fu, fv = feet_px(wk)
    k = CFG['WORKER_H'] / max(1.0, wk['box'][3] - wk['box'][1])        # pixel fallback: worker box height = 1.7 m
    return math.hypot(fu - (x1 + x2) / 2, fv - y2) * k, (x2 - x1) * k, 'pixel'

## 3. The three engines and the aggregator

In [ ]:
def struck_engine(wk, machines, ctx):
    if not machines:
        return dict(score=0.0, tier='no_machines_found', n_machines=0, nearest_m=None, nearest_box=None, mode=ctx['mode'])
    probs, best = [], None
    for m in machines:
        d, w, mode = worker_machine_dist(wk, m, ctx)
        probs.append(math.exp(-(d / max(CFG['MIN_RADIUS'], w)) ** 2))
        if best is None or d < best[0]: best = (d, mode, m['box'])
    score = 100 * (1 - np.prod([1 - s for s in probs]))
    tier = ctx['tier'] if best[1] == ctx['mode'] else 'low'
    return dict(score=float(score), tier=tier, n_machines=len(machines), nearest_m=float(best[0]), nearest_box=best[2], mode=best[1])

def head_prior(head_eff):
    """measured P(PPE really missing | no box found) for this head size (head height in px at detector resolution)"""
    edges = RC['head_bins']; i = next((k for k in range(len(edges) - 1) if edges[k] <= head_eff < edges[k + 1]), len(edges) - 2)
    return float(RC['prior_helmet'][i]), float(RC['prior_vest'][i])

def ppe_engine(wk, prox):
    e = wk['ppe']; evidence = any(e[k] > 0 for k in ('helmet', 'no_helmet', 'vest', 'no_vest'))
    if not evidence and wk['head_eff'] < RC['unverifiable_below_px']:
        return dict(score=None, m_helmet=None, m_vest=None, missing=[], status='unverifiable')
    pr_h, pr_v = head_prior(wk['head_eff'])
    m_h = (1 - e['helmet']) * (e['no_helmet'] + (1 - e['no_helmet']) * pr_h)
    m_v = (1 - e['vest']) * (e['no_vest'] + (1 - e['no_vest']) * pr_v)
    score = 100 * (CFG['W_HELMET'] * (1 + CFG['HELMET_PROX'] * prox) * m_h + CFG['W_VEST'] * (1 + prox) * m_v)
    return dict(score=float(min(100, score)), m_helmet=m_h, m_vest=m_v, status='evidence' if evidence else 'prior',
                missing=[n for n, m in (('helmet', m_h), ('vest', m_v)) if m > 0.5])

def fall_engine(wk, zones):
    """height above the ground (estimated elevation, optionally hand-drawn zones) x unstable posture"""
    ps, e = wk['posture'], wk['elev']
    sev_z, in_zone = 0.0, False
    fu, fv = feet_px(wk)
    for z in zones:
        if cv2.pointPolygonTest(np.array(z['polygon'], dtype=np.int32), (float(fu), float(fv)), False) >= 0:
            in_zone = True; sev_z = max(sev_z, min(1.0, z['drop_m'] / rp.CFG['ELEV_FULL_M']))
    sev = max(e['sev'] or 0.0, sev_z)
    score = 100 * (1 - (1 - CFG['W_HEIGHT'] * sev) * (1 - CFG['W_POSTURE'] * (ps['value'] or 0.0)))
    return dict(score=float(score), posture=ps['label'], in_zone=in_zone, severity=sev)

def band(r):
    return next(n for t, n in CFG['BANDS'] if r < t)

def aggregate(fall, ppe, struck, evidence):
    """evidence = (fall_has_evidence, ppe_has_evidence, struck_has_evidence): only engines with real evidence set the floor"""
    a = np.array([fall, ppe or 0.0, struck]) / 100
    base = CFG['W_FALL'] * a[0] + CFG['W_PPE'] * a[1] + CFG['W_STRUCK'] * a[2]
    pair = a[0] * a[1] + a[0] * a[2] + a[1] * a[2]
    ev_max = max([x for x, ok in zip(a, evidence) if ok] or [0.0])
    R = float(min(1.0, max(base + CFG['GAMMA'] * pair, CFG['FLOOR'] * ev_max)) * 100)
    return dict(risk=R, band=band(R))

def elevation_text(e):
    if e['kind'] == 'elevated': return f"elevated {e['m']:.1f} m"
    if e['kind'] == 'above_camera': return f"above camera level (>= {e['m']:.1f} m)"
    if e['kind'] == 'ground': return 'ground'
    if e['kind'] == 'below_ground': return f"below ground {-e['m']:.1f} m"
    return 'unknown'

## 4. Run one image with a chosen distance method

In [ ]:
def analyse_image(path):
    """the slow part (detector + pose + association + elevation) - done once per image, reused for both distance methods"""
    img = cv2.imread(path); stem = os.path.splitext(os.path.basename(path))[0]
    dets = detect(img)
    persons, pose_source = get_workers(img, stem, dets)
    cam, cam_note = load_camera(stem), ''
    if cam is not None and (cam['width'], cam['height']) != (img.shape[1], img.shape[0]):
        cam, cam_note = None, 'GeoCalib size differs from image'
    scene = rp.analyse_persons(persons, cam, img.shape, H_KNOWN.get(stem))
    for p_ in persons: p_['head_eff'] = rp.head_px(p_) * CFG['DET_IMGSZ'] / max(img.shape[:2])        # head height in px at detector resolution
    associate(persons, [d for d in dets if d['role'] in ('helmet', 'no_helmet', 'vest', 'no_vest')])
    return dict(stem=stem, img=img, persons=persons, machines=filter_machines(dets, img.shape), cam=cam, scene=scene,
                pose_source=pose_source, cam_note=cam_note)

def score_image(an, method):
    ctx = build_ctx(an, method)
    zones = ZONES.get(an['stem'], [])
    rows, extras = [], []
    for i, w in enumerate(an['persons']):
        sb = struck_engine(w, an['machines'], ctx)
        pp = ppe_engine(w, sb['score'] / 100)
        fl = fall_engine(w, zones)
        e = w['elev']
        ev = (e['kind'] != 'unknown' or fl['in_zone'], pp['status'] == 'evidence', sb['n_machines'] > 0)
        ag = aggregate(fl['score'], pp['score'], sb['score'], ev)
        n_ev = int(ev[1]) + int(ev[2]) + int(e['kind'] != 'unknown')
        rows.append(dict(image=an['stem'], worker=i, risk=round(ag['risk'], 1), band=ag['band'], risk_confidence=('high', 'medium', 'low')[0 if n_ev >= 2 else (1 if n_ev == 1 else 2)],
                         fall=round(fl['score'], 1), ppe=None if pp['score'] is None else round(pp['score'], 1), ppe_status=pp['status'], struck=round(sb['score'], 1),
                         missing=','.join(pp['missing']), posture=fl['posture'], elevation=elevation_text(e),
                         elev_m=None if e['m'] is None else round(e['m'], 2), elev_confidence=None if e['kind'] == 'unknown' else e['conf'],
                         nearest_m=None if sb['nearest_m'] is None else round(sb['nearest_m'], 2),
                         dist_mode=sb['mode'], dist_confidence=sb['tier'], n_machines=sb['n_machines'], worker_source=w['src'],
                         head_px_eff=round(w['head_eff'], 1)))
        extras.append(dict(feet=feet_px(w), nearest_box=sb['nearest_box']))
    return rows, extras, ctx

def agreement(rows_g, ctx_g, rows_d, ctx_d):
    """label-free check: do the two methods give the same nearest worker-machine distance?  -> % difference per worker"""
    out = []
    for rg, rd in zip(rows_g, rows_d):
        a, b = rg['nearest_m'], rd['nearest_m']
        ok = a is not None and b is not None and rg['dist_mode'] == 'geometry' and rd['dist_mode'] == 'depthpro' and (a + b) > 0
        out.append(round(100 * abs(a - b) / ((a + b) / 2), 1) if ok else None)
    return out

BAND_COL = {'Low': (60, 180, 75), 'Medium': (0, 200, 255), 'High': (0, 128, 255), 'Critical': (0, 0, 230)}   # BGR

def draw(an, rows, extras, ctx):
    vis = an['img'].copy(); H, W = vis.shape[:2]
    fs = max(0.5, W / 1600); th = max(1, int(round(W / 800)))
    for m in an['machines']:
        x1, y1, x2, y2 = [int(v) for v in m['box']]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 255), th + 1)
        cv2.putText(vis, m['name'], (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 0, 255), th)
    for w, r, ex in zip(an['persons'], rows, extras):
        x1, y1, x2, y2 = [int(v) for v in w['box']]; col = BAND_COL[r['band']]
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, th + 1)
        cv2.putText(vis, f"W{r['worker']} {r['risk']:.0f} {r['band']}", (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, fs, col, th)
        tag = f"F{r['fall']:.0f} P{'n/a' if r['ppe'] is None else format(r['ppe'], '.0f')} S{r['struck']:.0f}"
        if w['elev']['kind'] in ('elevated', 'above_camera'):
            tag += f" | UP {w['elev']['m']:.1f}m" if w['elev']['kind'] == 'elevated' else f" | UP >={w['elev']['m']:.1f}m"
        cv2.putText(vis, tag, (x1, min(H - 5, y2 + int(22 * fs))), cv2.FONT_HERSHEY_SIMPLEX, fs * 0.8, col, max(1, th - 1))
        if ex['nearest_box'] is not None and r['nearest_m'] is not None:
            bx = ex['nearest_box']; p0 = (int(ex['feet'][0]), int(ex['feet'][1])); p1 = (int((bx[0] + bx[2]) / 2), int(bx[3]))
            cv2.line(vis, p0, p1, col, th)
            cv2.putText(vis, f"{r['nearest_m']:.1f} m", ((p0[0] + p1[0]) // 2, (p0[1] + p1[1]) // 2), cv2.FONT_HERSHEY_SIMPLEX, fs, col, th)
    return vis

## 5. Run every image in the folder (both distance methods, so switching later is instant)

In [ ]:
paths = sorted(p for p in glob.glob(f'{IMG_DIR}/*') if p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(len(paths), 'images in', IMG_DIR)
assert paths, 'No images found - put them in the images folder first'
ANALYSES, RESULTS = {}, {}
for p in paths:
    an = analyse_image(p); stem = an['stem']; ANALYSES[stem] = an
    out = {m: score_image(an, m) for m in ('geometry', 'depthpro')}
    agr = agreement(out['geometry'][0], out['geometry'][2], out['depthpro'][0], out['depthpro'][2])
    for m in out:
        for r, a in zip(out[m][0], agr): r['agree_pct'] = a            # same value for both methods
    RESULTS[stem] = out
    kinds = [w['elev']['kind'] for w in an['persons']]
    print(f"{stem}: {len(an['persons'])} workers (pose: {an['pose_source']}), {len(an['machines'])} machines, "
          f"elevated={kinds.count('elevated') + kinds.count('above_camera')} | geometry -> {out['geometry'][2]['mode']}, depthpro -> {out['depthpro'][2]['mode']}")

## 6. Look at one image: pick the image and the distance method

In [ ]:
def show(stem, method):
    an = ANALYSES[stem]; rows, extras, ctx = RESULTS[stem][method]
    vis = draw(an, rows, extras, ctx)
    plt.figure(figsize=(11, 8)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'{stem}   |   distance method requested: {method}   |   used: {ctx["mode"]}'); plt.show()
    sc = an['scene']
    print(f"workers: pose source = {an['pose_source']} | elevation: {sc['status'] or 'not available'}"
          + (f" (camera height {sc['h_ref']:.2f} m from {sc['n_ref']} ground worker(s))" if sc['h_ref'] and sc['elev_ok'] else ''))
    if ctx['mode'] != method:
        print(f'NOTE: {method} not usable here ({ctx["note"] or "unknown reason"}) -> rough pixel-based distances (low confidence).')
    elif method == 'geometry':
        s_ = 'n/a' if ctx['spread'] is None else f"{ctx['spread']:.0%}"
        print(f"geometry: camera height {ctx['h']:.2f} m from {ctx['n_ref']} ground worker(s), spread {s_}")
    f1, f2 = ctx['f_geocalib'], ctx['f_depthpro']
    if f1 and f2:
        print(f'focal length: GeoCalib {f1:.0f} px vs Depth Pro {f2:.0f} px (ratio {f2 / f1:.2f})')
        if not 0.7 <= f2 / f1 <= 1.4: print('WARNING: the two methods disagree strongly on the focal length - treat distances in this image with care.')
    ag = [r['agree_pct'] for r in rows if r['agree_pct'] is not None]
    if ag:
        print(f"method agreement on the nearest worker-machine distance: median difference {np.median(ag):.0f}% -> "
              f"{'agree (higher confidence)' if np.median(ag) <= CFG['AGREE_TOL'] else 'DISAGREE (treat distances with care)'}")
    cols = ['worker', 'risk', 'band', 'risk_confidence', 'fall', 'ppe', 'ppe_status', 'struck', 'missing', 'posture', 'elevation', 'nearest_m', 'dist_mode', 'dist_confidence', 'n_machines', 'agree_pct', 'worker_source']
    display(pd.DataFrame(rows)[cols] if rows else pd.DataFrame(columns=cols))

# UI
import ipywidgets as w
from IPython.display import display, clear_output
img_dd = w.Dropdown(options=list(ANALYSES), description='Image')
meth_tb = w.ToggleButtons(options=[('Geometry (GeoCalib)', 'geometry'), ('Depth Pro', 'depthpro')], description='Distance')
panel = w.Output()
def _update(*_):
    with panel:
        clear_output(wait=True)
        if img_dd.value: show(img_dd.value, meth_tb.value)
img_dd.observe(_update, 'value'); meth_tb.observe(_update, 'value')
display(w.VBox([img_dd, meth_tb, panel])); _update()

## 7. Save everything (tables + annotated images for both methods)

In [ ]:
all_rows = []
for stem, out in RESULTS.items():
    for m, (rows, extras, ctx) in out.items():
        cv2.imwrite(f'{OUT_DIR}/{stem}__{m}.jpg', draw(ANALYSES[stem], rows, extras, ctx))
        all_rows += [dict(r, method_requested=m, method_used=ctx['mode'], note=ctx['note']) for r in rows]
res = pd.DataFrame(all_rows)
res['risk_percentile'] = (res.groupby('method_requested')['risk'].rank(pct=True) * 100).round(0)       # rank of this worker among all workers in the folder
res.to_csv(f'{OUT_DIR}/results.csv', index=False); res.to_json(f'{OUT_DIR}/results.json', orient='records', indent=1)
print('saved', len(res), 'rows and', 2 * len(RESULTS), 'annotated images to', OUT_DIR)
res.sort_values('risk', ascending=False).head(15)

## 8. OPTIONAL - bands from your own results (quantiles)
Replaces the placeholder 25 / 50 / 75 cut-offs with the 50th / 75th / 90th percentile of the risk scores in this folder (needs at least `MIN_WORKERS` workers; use a folder of many typical photos, e.g. the `accuracy` set). The bands then mean "above the median / top quarter / top 10% of what this system sees". Saved into `risk_config.json`; the next run of this notebook uses them.

In [ ]:
WRITE_BANDS = False          # set True to save
MIN_WORKERS = 30
r_ = res[res.method_requested == 'depthpro'].risk
if len(r_) < MIN_WORKERS:
    print(f'only {len(r_)} workers here - need {MIN_WORKERS} for stable quantiles; keeping the current bands')
else:
    edges = [round(float(r_.quantile(q)), 1) for q in (0.5, 0.75, 0.9)]
    print('quantile band edges (Low | Medium | High | Critical):', edges)
    if WRITE_BANDS:
        rc = json.load(open(CONFIG_PATH)) if os.path.exists(CONFIG_PATH) else dict(DEFAULTS)
        rc['band_edges'] = edges; json.dump(rc, open(CONFIG_PATH, 'w'), indent=1); print('saved to', CONFIG_PATH)